# 05 - Fase 3: GAN condicional por hidrofobicidad

Este notebook entrena una **GAN condicional por hidrofobicidad**: se le puede pedir la hidrofobicidad media del péptido (el GRAVY de Kyte-Doolittle). Es el segundo de los tres experimentos condicionales previstos (longitud en el notebook 04, **hidrofobicidad** y ambas a la vez). Se entrena con **el mismo protocolo** que las variantes del notebook 02 (datos de **Antibacterial**, `Antibacterial_amps_clean.fasta`, 5-60 aa; 200 épocas; instantáneas en las épocas 1, 20, 40, ..., 200; 2000 secuencias por instantánea; semilla 42; mismas métricas y mismos dos modos de decodificación, **sample** como principal y **argmax** como secundario) y se compara con resultados de referencia ya generados, que **no se reentrenan**: se leen de `context/data/processed/`.

**Referencia de la Fase 3:** la GAN vanilla estabilizada con ruido de instancia constante (`gan_stabc`, sección h del notebook 02). Las otras dos variantes del 02 (`gan_base` y `gan_stab`), la WGAN-GP del notebook 03 (`wgan_gp_*`) y la condicional por longitud del notebook 04 (`cgan_length_*`) se muestran **solo como información**.

**Relación con los otros notebooks.** Es autocontenido: no necesita ejecutar el 03 ni el 04 (carga sus resultados guardados). El criterio A.2 de este experimento se **revisó después de conocer el resultado**; el criterio inicial se conserva y se muestra junto al revisado (secciones 6.1 y 6.11).

Orden: preparación (sección 1), entorno (2), elección del backend y dispositivo de Keras (3; por defecto CPU, opcionalmente una prueba de rendimiento), datos (4), resultados de referencia (5) y GAN condicional por hidrofobicidad (6).

> **Tiempo de ejecución.** La GAN condicional por hidrofobicidad tiene el mismo coste por época que la condicional por longitud, parecido al de la vanilla estabilizada con ruido constante. Medido en la máquina de desarrollo (RTX 3060 portátil y CPU de 16 hilos) ejecutando este notebook completo con `EVALUAR_BACKEND = True` (torch en GPU): **571 s en total (~9,5 min)**, de los cuales 42 s son la prueba de rendimiento de la celda opcional de la sección 3, 510 s el entrenamiento (2,55 s por época de media) y 9 s las evaluaciones por instantánea. Con el valor por defecto (`EVALUAR_BACKEND = False`: CPU con TensorFlow) se midieron **3,92 s por época** (mediana de 6 épocas completas), de modo que las 200 épocas serían **~13 min (estimado a partir de ese tiempo por época; no se ha ejecutado el notebook entero en CPU)**. Cargar las referencias y dibujar es inmediato.
>
> **Reproducibilidad.** Todas las semillas están fijadas, pero los números pueden variar entre backends y dispositivos (las operaciones en coma flotante no dan resultados idénticos en TensorFlow, torch, CPU o GPU). Los CSV de referencia, los de `wgan_gp_*`, `cgan_length_*` y `cgan_hydro_*` (scripts `run_gan_base.py` y `run_conditional_gan.py`) en `context/data/processed/` se generaron con **torch en GPU**; al ejecutar en CPU las curvas y métricas serán parecidas pero no idénticas, y los números concretos que cita el texto se generan a partir de la ejecución en curso. Es **una sola ejecución con la semilla 42**: no se estima la variabilidad entre semillas.

La lógica vive en `src/amp_gan/` (`conditional_gan`, `conditions`, `experiment`, `evaluation`, `properties`, `runtime`); aquí solo se importa, se ejecuta y se dibuja. Las métricas son las del notebook 02: longitud, composición de aminoácidos, carga neta (pH 7,4) e hidrofobicidad media de Kyte-Doolittle (GRAVY).

## 1. Preparación: rutas, semilla y versiones

In [ ]:
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

# Raíz del proyecto: se sube desde el directorio actual hasta encontrar `context/` y `src/` (sin rutas absolutas).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "context").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Importamos solo módulos que NO cargan Keras (el backend debe fijarse antes de importarlo).
from amp_gan import runtime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROCESSED = ROOT / "context" / "data" / "processed"
TRAIN_FASTA = PROCESSED / "Antibacterial_amps_clean.fasta"
BATCH_SIZE = 128
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("python    :", platform.python_version())
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("scipy     :", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("semilla   :", SEED)
print("keras ya importado en este kernel:", "keras" in sys.modules)

## 2. Entorno

Versiones de las librerías de aprendizaje profundo (leídas de los metadatos del paquete, sin importarlas) y GPU que ve el sistema. Todavía no se ha importado Keras: su backend y su dispositivo se eligen en la sección siguiente.

In [ ]:
import importlib.metadata as md_
import os
import subprocess

def installed_version(*names):
    for name in names:
        try:
            return f"{md_.version(name)} (paquete {name})"
        except md_.PackageNotFoundError:
            pass
    return "no instalado"

for label, names in (("keras", ("keras", "keras-nightly")), ("tensorflow", ("tensorflow",)), ("torch", ("torch",))):
    print(f"{label:11s}: {installed_version(*names)}")
print("backends instalados para Keras:", runtime.installed_backends())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20, check=True).stdout.strip()
    print("GPU (nvidia-smi):", gpu)
except Exception as exc:
    print("GPU (nvidia-smi): no disponible:", type(exc).__name__)

## 3. Backend y dispositivo de Keras

Keras 3 puede ejecutarse sobre TensorFlow o torch, en CPU o en GPU. Keras lee el backend (`KERAS_BACKEND`) y TensorFlow/torch leen la visibilidad de la GPU (`CUDA_VISIBLE_DEVICES`) **solo al importarse**, así que la elección debe hacerse antes de importar Keras (hasta aquí no se ha importado). Si el kernel ya hubiese importado Keras con otro backend, habría que reiniciarlo.

**Por defecto se usa la CPU**, con TensorFlow si está instalado (fue el más rápido en CPU) y, si no, torch. Así el notebook funciona en cualquier máquina, sin GPU y sin depender de archivos generados antes. Se puede forzar el backend con la variable de entorno `AMP_GAN_BACKEND` (`tensorflow` o `torch`) antes de arrancar el kernel.

La GAN condicional usa solo la API pública de Keras (sin código específico de backend), así que funciona con TensorFlow y con torch.

In [ ]:
decision = runtime.apply_cpu_default()
assert "keras" not in sys.modules, "Keras ya está importado: reinicia el kernel antes de elegir el backend."
print(decision["message"])

### Opcional: ¿hay un backend/dispositivo más rápido en tu máquina?

> **Opcional: evaluar si hay un backend/dispositivo más rápido; si no dispones de gráfica puedes saltarla y se usa la CPU.**

Con `EVALUAR_BACKEND = True` la celda siguiente mide el paso de entrenamiento de la GAN base vanilla (mismo modelo, mismos datos, misma semilla; la GAN condicional tiene un coste por época parecido, algo mayor, pero la decisión de backend se toma con esta misma prueba, que es la del resto del proyecto) para cada configuración `(backend, dispositivo)` **disponible en esta máquina** (solo se prueban los backends instalados y la GPU solo si se detecta) y varios tamaños de lote, y aplica la más rápida. Cada configuración se ejecuta en un subproceso independiente para que el backend y la GPU queden bien aislados, y una configuración que falle se descarta sin interrumpir el resto. Se miden menos pasos que en la prueba completa de `python -m amp_gan.benchmark` (60 pasos medidos y 20 de calentamiento, frente a 200 y 20), de modo que las cifras son algo más ruidosas; el resultado se devuelve en memoria y no se escribe a disco ni se lee ningún archivo de otra máquina.

Si no hay ninguna GPU utilizable, la prueba no se ejecuta y se mantiene la CPU. Con el valor por defecto (`False`) esta sección no hace nada más.

In [ ]:
EVALUAR_BACKEND = False  # True: medir y aplicar el backend/dispositivo más rápido (opcional; requiere GPU para que merezca la pena)

bench = None
if EVALUAR_BACKEND:
    t0 = time.perf_counter()
    decision, bench = runtime.apply_auto_backend(steps=60, warmup=20)
    print(f"Evaluación hecha en {time.perf_counter() - t0:.0f} s")
    print(decision["message"])
    if bench is not None:
        view = bench[["backend", "device", "batch_size", "status", "ms_per_step_median", "samples_per_s", "est_epoch_s", "error"]].copy()
        view["error"] = view["error"].fillna("").str.slice(0, 90)
        display(view.round(3).reset_index(drop=True))
else:
    print("Evaluación omitida (EVALUAR_BACKEND = False): se mantiene la configuración por defecto.")
    print(decision["message"])

Solo si se ha evaluado: tiempo estimado por época (segundos; menos es mejor) por configuración y tamaño de lote. Las configuraciones que fallan no tienen barra: se marcan como *no disponible* en la leyenda y su motivo se resume debajo del gráfico. Las configuraciones no disponibles en la máquina (por ejemplo, TensorFlow en GPU si no se detecta) ni siquiera se prueban.

In [ ]:
if bench is not None:
    configs = list(dict.fromkeys(zip(bench["backend"], bench["device"])))
    batch_sizes = sorted(bench["batch_size"].unique())
    colors = {("tensorflow", "cpu"): "tab:blue", ("torch", "cpu"): "tab:green",
              ("torch", "gpu"): "tab:red", ("tensorflow", "gpu"): "tab:purple"}

    fig, ax = plt.subplots(figsize=(10, 4.2))
    width = 0.8 / len(configs)
    for i, (backend, device) in enumerate(configs):
        sub = bench[(bench["backend"] == backend) & (bench["device"] == device)].set_index("batch_size")
        ok = (sub["status"] == "ok").any()
        label = f"{backend}/{device}" + ("" if ok else " (no disponible)")
        x = np.arange(len(batch_sizes)) + (i - (len(configs) - 1) / 2) * width
        vals = [sub.loc[b, "est_epoch_s"] if b in sub.index and sub.loc[b, "status"] == "ok" else np.nan for b in batch_sizes]
        bars = ax.bar(x, np.nan_to_num(vals), width=width, color=colors.get((backend, device), "gray"), label=label)
        for rect, v in zip(bars, vals):
            if np.isfinite(v):
                ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(np.arange(len(batch_sizes)))
    ax.set_xticklabels(batch_sizes)
    ax.set_xlabel("Tamaño de lote")
    ax.set_ylabel("Tiempo estimado por época (s)")
    ax.set_title("Prueba de rendimiento de la GAN base: tiempo por época")
    ax.legend(title="backend/dispositivo")
    fig.tight_layout()
    plt.show()

    errors = bench[bench["status"] != "ok"].drop_duplicates(["backend", "device"])
    for _, r in errors.iterrows():
        print(f"No disponible: {r['backend']}/{r['device']} -> {str(r['error'])[:200]}")

La regla de decisión del proyecto, para el tamaño de lote de entrenamiento (128), es: gana la configuración con menor tiempo estimado por época; si la segunda está a menos de un 10 % se declara empate y se prefiere la opción más simple (CPU antes que GPU, TensorFlow antes que torch). El backend y el dispositivo ya están fijados en el entorno, **antes de importar Keras**; la celda siguiente muestra la configuración vigente.

In [ ]:
assert "keras" not in sys.modules, "Keras ya está importado: reinicia el kernel antes de elegir el backend."
print(f"Configuración aplicada: modo {decision['mode']}, backend={decision['backend']}, dispositivo={decision['device']}")

Ahora sí se importa Keras (a través de `describe_environment`) y se confirma el backend y el dispositivo realmente usados.

In [ ]:
env = runtime.describe_environment()
for k in ("keras_backend", "keras_devices", "gpu_visible", "cuda_visible_devices", "keras_version"):
    print(f"{k:22s}: {env[k]}")
assert env["keras_backend"] == decision["backend"]
assert env["gpu_visible"] == (decision["device"] == "gpu")

## 4. Datos

Se cargan las secuencias limpias de Antibacterial (5-60 aa, 20 aminoácidos estándar, sin duplicados exactos) y se codifican en one-hot con relleno `PAD` hasta 60 posiciones.

In [ ]:
from amp_gan.encoding import SEQ_LEN, VOCAB_SIZE, load_encoded

X, df = load_encoded(TRAIN_FASTA)
print(f"secuencias: {len(df):,}")
print(f"X: forma {X.shape}, dtype {X.dtype}, tamaño {X.nbytes / 2**20:.0f} MiB")
print(f"seq_len = {SEQ_LEN}, vocabulario = {VOCAB_SIZE} (20 aa + PAD)")

## 5. Resultados de referencia (sin reentrenar)

Se cargan de `context/data/processed/` las métricas, el historial, la información de la ejecución y las 2000 secuencias de la época final de las tres variantes del notebook 02, de la WGAN-GP del notebook 03 y de la condicional por longitud del notebook 04. Si falta algún archivo, la celda se detiene con el comando que lo genera. Estas ejecuciones se hicieron con torch en GPU (ver `run_info.json`).

In [ ]:
import json

from amp_gan.data import read_fasta
from amp_gan.experiment import MODES

EPOCHS, SNAPSHOT_EVERY, N_EVAL = 200, 20, 2000  # el protocolo de todo el proyecto

REF_NAME = "estab. ruido constante (referencia)"
COND_NAME = "condicional (longitud)"
REFERENCIAS = {"vanilla pura": "gan_base", "estabilizada": "gan_stab", REF_NAME: "gan_stabc", "WGAN-GP": "wgan_gp",
               COND_NAME: "cgan_length"}
COMANDOS = {"gan_base": "python src/amp_gan/run_gan_base.py --variant base",
            "gan_stab": "python src/amp_gan/run_gan_base.py --variant stabilized",
            "gan_stabc": "python src/amp_gan/run_gan_base.py --variant stabilized_const",
            "wgan_gp": "python src/amp_gan/run_gan_base.py --variant wgan_gp",
            "cgan_length": "python src/amp_gan/run_conditional_gan.py --variant cgan_length --auto-backend"}
SUFIJOS = ("metrics.csv", "history.csv", "run_info.json", "generated_sample.fasta", "generated_argmax.fasta")

faltan = [f"{p}_{s}" for p in REFERENCIAS.values() for s in SUFIJOS if not (PROCESSED / f"{p}_{s}").exists()]
if faltan:
    prefijos = sorted({"_".join(f.split("_")[:2]) for f in faltan})
    comandos = "\n".join(f"    {COMANDOS[p]}" for p in prefijos)
    raise FileNotFoundError(
        "Faltan resultados de referencia en context/data/processed/ (" + ", ".join(faltan) + ").\n"
        "Genéralos desde la raíz del proyecto con:\n" + comandos)

ref_metrics, ref_hist, ref_info, ref_seqs = {}, {}, {}, {}
for name, prefix in REFERENCIAS.items():
    m = pd.read_csv(PROCESSED / f"{prefix}_metrics.csv")
    ref_metrics[name] = m[m["epoch"].notna()].astype({"epoch": int})  # las filas de suelo no tienen época
    ref_hist[name] = pd.read_csv(PROCESSED / f"{prefix}_history.csv")
    ref_info[name] = json.loads((PROCESSED / f"{prefix}_run_info.json").read_text(encoding="utf-8"))
    ref_seqs[name] = {mode: read_fasta(PROCESSED / f"{prefix}_generated_{mode}.fasta")["sequence"].tolist() for mode in MODES}

# Las cinco variantes guardadas, en el orden de las tablas (la condicional por longitud, solo como información).
ALL_METRICS, ALL_SEQS = ref_metrics, ref_seqs
COLORS = {"vanilla pura": "tab:red", "estabilizada": "tab:green", REF_NAME: "tab:blue", "WGAN-GP": "tab:purple",
          COND_NAME: "tab:orange"}

resumen = []
for name, prefix in REFERENCIAS.items():
    info, hh = ref_info[name], ref_hist[name]
    steps = hh["generator_steps"].mean() if "generator_steps" in hh else info["n_train"] // info["config"]["batch_size"]
    resumen.append({"variante": name, "archivos": prefix + "_*", "épocas": len(hh), "backend": info["keras_backend"],
                    "dispositivo": info["device_used"], "s/época": info["timing_s"]["mean_epoch_s"],
                    "actualizaciones de G por época": steps})
display(pd.DataFrame(resumen).set_index("variante").round(2))

Utilidades comunes de las secciones siguientes: los suelos de referencia (`real_subset`: lo mínimo alcanzable por el ruido del muestreo; `random_uniform`: ningún aprendizaje), las columnas de las tablas de métricas, la función que toma la fila de una época y un modo, la lista de propiedades de los histogramas, la tabla de secuencias de ejemplo y las demás funciones auxiliares de las tablas y gráficos. Mismas definiciones que en el notebook 03: fracción de secuencias válidas, únicas y nuevas; media, desviación típica y W1 (distancia de Wasserstein-1 frente a las secuencias reales) de longitud, carga neta e hidrofobicidad; y `composition_tv`, la variación total entre la composición media de aminoácidos generada y la real.

In [ ]:
from collections import Counter

from matplotlib.lines import Line2D

from amp_gan.data import STANDARD_ALPHABET
from amp_gan.evaluation import is_valid, profile_sequences, reference_floors
from amp_gan.properties import properties_table

floors = reference_floors(df, n=N_EVAL, seed=SEED)
props_real = properties_table(df)
prof_real = profile_sequences(df["sequence"].tolist())
real_mean_len = float(props_real["length"].mean())
cols = ["valid_fraction", "unique_fraction", "novel_fraction", "length_mean", "length_std", "length_wasserstein",
        "net_charge_mean", "net_charge_std", "net_charge_wasserstein",
        "hydrophobicity_mean", "hydrophobicity_std", "hydrophobicity_wasserstein", "composition_tv"]

def pick(metrics_df, mode, epoch=EPOCHS):
    return metrics_df[(metrics_df["epoch"] == epoch) & (metrics_df["mode"] == mode)].iloc[0]

def snap_series(metrics_df, col, mode="sample"):
    return metrics_df[metrics_df["mode"] == mode].set_index("epoch")[col]

PROPS = [("length", "Longitud (aa)"), ("net_charge", "Carga neta (pH 7,4)"), ("hydrophobicity", "Hidrofobicidad (GRAVY)")]
panels = [
    ("unique_fraction", "Fracción de secuencias únicas", None),
    ("length_wasserstein", "W1 de longitud (aa)", None),
    ("net_charge_wasserstein", "W1 de carga neta", None),
    ("hydrophobicity_wasserstein", "W1 de hidrofobicidad", None),
    ("composition_tv", "Variación total de composición", None),
    ("length_mean", "Longitud media (aa)", real_mean_len),
]

def example_table(seqs, k=10):
    rows = []
    for s in seqs[:k]:
        if is_valid(s):
            p = profile_sequences([s])
            rows.append({"sequence": s, "length": len(s), "net_charge": p.net_charge[0], "hydrophobicity": p.hydrophobicity[0]})
        else:
            rows.append({"sequence": s if s else "(vacía)", "length": len(s), "net_charge": np.nan, "hydrophobicity": np.nan})
    return pd.DataFrame(rows).round(2)

pd.set_option("display.max_colwidth", 80)
print("Referencia real (Antibacterial completo):")
display(pd.DataFrame({"media": props_real[["length", "net_charge", "hydrophobicity"]].mean(),
                      "desv. típica": props_real[["length", "net_charge", "hydrophobicity"]].std()}).round(3))

## 6. GAN condicional por hidrofobicidad

### 6.1 Qué cambia y por qué

Es el segundo de los tres experimentos condicionales (longitud, **hidrofobicidad** y ambas a la vez). Con la misma base y el mismo protocolo que el notebook 04 (condicional por longitud), ahora se le puede pedir al generador *«un péptido con hidrofobicidad media X»*.

**Por qué la hidrofobicidad.** Los péptidos antimicrobianos actúan en buena medida sobre la membrana bacteriana y lo hacen por su **anfipaticidad**: un balance entre residuos catiónicos, que los acercan a una membrana cargada negativamente, y residuos hidrofóbicos, que se insertan en la bicapa lipídica. La hidrofobicidad media modula ese balance: con muy poca, el péptido interacciona poco con la membrana; con demasiada, suele perder selectividad (también puede dañar membranas de células del huésped) y tiende a agregarse. Poder fijarla en el generador permite explorar ese compromiso (más adelante, junto con la longitud). Aquí solo se mide si el generador **obedece** la condición; no se evalúa la actividad de las secuencias.

**Base:** la misma que en el notebook 04, la vanilla estabilizada con ruido de instancia constante (`GANConfig.stabilized_constant_noise()`, la referencia de la Fase 3): etiqueta real 0,9, ruido de instancia constante de desviación 0,1, Gumbel-softmax *straight-through* con τ = 1, generador MLP (256, 512), discriminador MLP (512, 256), `z` de 100, Adam con tasa 2e-4 y β1 = 0,5, lote 128, semilla 42. **No se cambia ningún hiperparámetro.** Solo cambia la condición:

- **Condición.** La hidrofobicidad es el **GRAVY**: la media de la escala de hidropatía de Kyte-Doolittle sobre todos los residuos (valores altos, más hidrofóbico; poli-I = 4,5 y poli-R = -4,5). Se normaliza a `[0, 1]` con un min-max con el mínimo y el máximo de las secuencias de **entrenamiento**: `(g - min) / (max - min)`. A diferencia de la longitud (rango fijo 5-60 aa), los extremos dependen de los datos; se muestran al entrenar. La condición se entrena con **todos** los valores de GRAVY; el rango habitual solo define lo que se pide al generar.
- **Generador `[z, c]` y discriminador `[secuencia, c]`.** Igual que en el notebook 04, con la condición (de dimensión 1) concatenada a `z` (101 entradas) y a la secuencia aplanada (1 261 entradas); el ruido de instancia no se aplica a la condición.
- **Entrenamiento.** Cada lote lleva la **condición real** de sus secuencias (su GRAVY verdadero, normalizado), que se usa para el lote real, para generar el falso y para el paso del generador.
- **Generación.** Se pide una condición y se decodifica con `sample_sequences` (modos `sample` y `argmax`).

### Rango «habitual» y objetivos de hidrofobicidad

Lo que se pide al generar y se evalúa es el **rango habitual de GRAVY: percentiles 10 a 90 de las secuencias reales de Antibacterial**. Se evalúan cinco objetivos continuos, los percentiles 10, 30, 50, 70 y 90 del GRAVY real (**valores exactos, sin redondear**). La **tolerancia de acierto** está fijada de antemano en **±0,25** unidades de GRAVY. La celda siguiente calcula todo de los datos.

### Dos evaluaciones

- **(i) Evaluación «empírica».** Las condiciones son los GRAVY de secuencias reales muestreadas con reposición del conjunto de entrenamiento (semilla 42), con las mismas métricas y formato que las demás variantes, y por tanto comparable con la referencia sin condicionar.
- **(ii) Evaluación por objetivos.** Para cada objetivo `t` se piden 2000 secuencias y se mide el GRAVY medio generado (sobre las secuencias no vacías: una cadena vacía no tiene GRAVY), el error absoluto medio `|GRAVY - t|`, la **tasa de acierto** (fracción de **todas** las secuencias con `|GRAVY - t| ≤ 0,25`; una cadena vacía es un fallo) y la fracción de válidas y de únicas. Como propiedades **secundarias**, la W1 de **longitud** y de **carga neta** de las secuencias generadas válidas frente a los datos reales con GRAVY en `[t - 0,25, t + 0,25]` (si ese subconjunto tiene menos de 30 secuencias, NaN y se indica). Referencias **sin condición**: (a) la muestra de la referencia (`gan_stabc`, 2000 secuencias), que no se puede dirigir: tasa de acierto ±0,25 para cada `t` y GRAVY medio de toda la muestra; (b) el **techo real**: fracción de los datos reales con `|GRAVY - t| ≤ 0,25`.

### Criterios fijados de antemano (modo sample, época 200, sin elegir la mejor época)

**A. Control de la hidrofobicidad.** La condicional controla la hidrofobicidad si cumple las tres condiciones:
1. la pendiente de la regresión lineal de la GRAVY media generada frente a la GRAVY pedida (los cinco objetivos) está entre 0,8 y 1,2;
2. el error absoluto medio `|GRAVY - t|`, promediado sobre los cinco objetivos, es ≤ 0,30;
3. su tasa de acierto (±0,25) supera a la de la referencia en los cinco objetivos.

**B. Fidelidad con condiciones reales** (evaluación (i)). La condicional **no empeora a la referencia** si su W1 de hidrofobicidad es ≤ 0,0848 (la de la referencia en la época 200) y su `unique_fraction` es ≥ 0,95. Las W1 de longitud y de carga neta y la variación total de composición se informan por separado y de forma descriptiva (sin condición de aprobado), junto a las de la referencia y a las de la condicional por longitud.

Se informa de cada condición por separado y del resultado global de A y de B, sea cual sea; no se ajustan hiperparámetros ni se repite con otra semilla. Es **una sola ejecución con la semilla 42**. Si los objetivos extremos (p10, p90) quedan cortos aunque haya pocas secuencias reales cerca, se documenta en lugar de corregirlo.

**Revisión posterior del criterio A.2 (hecha después de conocer el resultado).** El criterio A.2 de arriba (error medio ≤ 0,30) **se modificó después de ver el resultado de este experimento**: no se cumplió (error medio 0,558) y, al analizarlo, resultó inalcanzable por construcción, porque la desviación típica del GRAVY generado en cada objetivo es ≈ 0,7 y con esa dispersión el error absoluto medio ya es ≈ 0,55 aunque la media generada fuese exacta (detalle y cifras en 6.11). Se conserva el criterio inicial, que se sigue mostrando como «criterio inicial», y se añade el **criterio revisado**:

2'. (A.2 revisado) el error absoluto medio `|GRAVY - t|` del modelo es **menor que el de la referencia sin condición (`gan_stabc`) en cada uno de los cinco objetivos** (sin umbral numérico; misma lógica que A.3).

Los criterios A.1, A.3 y B no cambian. Como este cambio es posterior al resultado, el veredicto de A con el criterio revisado es **menos riguroso** que el de un criterio fijado de antemano y se informa siempre junto al del criterio inicial.

In [ ]:
from amp_gan.conditions import (HYDRO_HIT_TOLERANCE, TARGET_PERCENTILES, unconditional_target_table, control_slope, property_percentiles, property_values, real_ceiling,
                                real_subset_around, target_values, usual_range)

def fmt(x, d=3):
    # Número con coma decimal para las frases generadas.
    return f"{np.round(x, d):.{d}f}".replace(".", ",")  # mismo redondeo que las tablas (pandas)

real_gravy = property_values(df["sequence"], "hydrophobicity")
H_PERCENTILES = property_percentiles(real_gravy)
H_TARGETS = target_values(real_gravy, "hydrophobicity")  # p10, p30, p50, p70, p90: valores reales, sin redondear
H_LO, H_HI = usual_range(real_gravy)
H_TOL = HYDRO_HIT_TOLERANCE
print(f"GRAVY real: n = {real_gravy.size}, media {fmt(real_gravy.mean())}, desviación típica {fmt(real_gravy.std(ddof=1))}, "
      f"mínimo {fmt(real_gravy.min())}, máximo {fmt(real_gravy.max())}.")
print(f"Rango habitual (percentiles 10-90): de {fmt(H_LO)} a {fmt(H_HI)}. Tolerancia de acierto: +-{fmt(H_TOL, 2)}.")
h_ceiling = real_ceiling(real_gravy, H_TARGETS, H_TOL).set_index("target")["hit_rate"]
h_near = {t: len(real_subset_around(df["sequence"], t, H_TOL, prop="hydrophobicity")) for t in H_TARGETS}
tabla_h = pd.DataFrame({
    "percentil": [f"p{p}" for p in TARGET_PERCENTILES],
    "objetivo t (GRAVY)": list(H_TARGETS),
    f"reales con |GRAVY-t| <= {fmt(H_TOL, 2)}": [h_near[t] for t in H_TARGETS],
    "techo real (tasa de acierto)": [h_ceiling[t] for t in H_TARGETS],
}).set_index("percentil")
display(tabla_h.round(3))

### 6.2 Entrenamiento

Se usa `ConditionalGANConfig.stabilized_constant_noise(conditions=("hydrophobicity",))` (la configuración de la referencia más la condición) y el mismo `run_conditional_experiment` del notebook 04, ahora con `target_property="hydrophobicity"`: 200 épocas, instantáneas en la 1 y cada 20, 2000 secuencias por instantánea con ruido fijo (semilla 42) y modos `argmax` y `sample`; en cada instantánea se hacen las dos evaluaciones, (i) y (ii). `MAX_PASOS` sigue en `None` (una pasada completa por los datos en cada época). La primera época incluye la construcción (y, con TensorFlow, la compilación) del modelo; el tiempo de entrenamiento no incluye el de las evaluaciones.

In [ ]:
from amp_gan.conditional_gan import ConditionalGANConfig, run_conditional_experiment

MAX_PASOS = None  # None: una pasada completa por los datos en cada época (solo se cambia para pruebas rápidas)
COND_H = "condicional (hidrofobicidad)"
config_h = ConditionalGANConfig.stabilized_constant_noise(seed=SEED, conditions=("hydrophobicity",))
print(config_h)

res_h = run_conditional_experiment(X, df, config_h, epochs=EPOCHS, snapshot_every=SNAPSHOT_EVERY, n_eval=N_EVAL,
                                   eval_seed=SEED, targets=H_TARGETS, tol=H_TOL, max_steps_per_epoch=MAX_PASOS,
                                   target_property="hydrophobicity")
t_h = res_h.timing
spec_h = res_h.history.spec
print(f"Condición: {spec_h.names}, normalización min-max con extremos {fmt(spec_h.low[0])} y {fmt(spec_h.high[0])} (GRAVY de las secuencias de entrenamiento).")
print("Objetivos normalizados: " + ", ".join(fmt(v) for v in spec_h.normalize(np.array(H_TARGETS)[:, None])[:, 0]) + ".")
print(f"Tiempo total: {t_h['total_s']:.1f} s (entrenamiento {t_h['train_s']:.1f} s + evaluaciones {t_h['eval_s']:.1f} s)")
print(f"Por época: media {t_h['mean_epoch_s']:.2f} s, mediana {t_h['median_epoch_s']:.2f} s "
      f"(backend {env['keras_backend']}, dispositivo {decision['device']})")

### 6.3 Curvas de pérdida

Las tres medidas por época de siempre, de la condicional por hidrofobicidad (continua) y de la referencia sin condicionar (discontinua). Con la etiqueta real suavizada a 0,9 la pérdida del discriminador con lotes reales no puede bajar de ~0,33.

In [ ]:
epochs_axis_c = np.arange(1, EPOCHS + 1)
panels_c = [("d_loss_real", "Discriminador, lote real"), ("d_loss_fake", "Discriminador, lote falso"), ("g_loss", "Generador")]
h_h = res_h.history
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (key, title) in zip(axes, panels_c):
    ax.plot(epochs_axis_c, h_h[key], color="tab:brown", label=COND_H)
    ax.plot(ref_hist[REF_NAME]["epoch"], ref_hist[REF_NAME][key], color=COLORS[REF_NAME], ls="--", lw=1, label=REF_NAME)
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

hist_h = pd.DataFrame({k: h_h[k] for k in ("d_loss_real", "d_loss_fake", "d_loss", "g_loss")}, index=epochs_axis_c)
display(pd.DataFrame({"primeras 5 épocas": hist_h.head(5).mean(), "épocas 96-100": hist_h.loc[96:100].mean(),
                      "últimas 5 épocas": hist_h.tail(5).mean()}).round(4))
print("¿Algún valor no finito en el historial?", bool((~np.isfinite(hist_h.to_numpy(dtype=float))).any()))

### 6.4 Métricas finales de la evaluación (i) (época 200)

Mismo formato y columnas que las tablas 5.5 del notebook 03 y 6.4 del notebook 04. La **referencia** (estabilizada con ruido constante) es la comparación del criterio B; la WGAN-GP (notebook 03) y la condicional por longitud (notebook 04, evaluada con longitudes reales), ambas leídas de sus resultados guardados, se muestran **solo como información**. La condicional por hidrofobicidad se evalúa con GRAVY tomados de secuencias reales. Suelos: `real_subset` (lo mínimo alcanzable por el ruido del muestreo) y `random_uniform` (ningún aprendizaje).

In [ ]:
METRICS_C = {**ALL_METRICS, COND_H: res_h.metrics}
SEQS_C = {**ALL_SEQS, COND_H: {mode: res_h.sequences[mode] for mode in MODES}}
COLORS_C = {**COLORS, COND_H: "tab:brown"}
COND_INFO = f"{COND_NAME} [informativa]"

def final_table_c(mode):
    table = pd.DataFrame({(COND_INFO if name == COND_NAME else name): pick(m, mode) for name, m in METRICS_C.items()}
                         | {"suelo real_subset": pd.Series(floors["real_subset"]),
                            "suelo random_uniform": pd.Series(floors["random_uniform"])})
    return table.loc[cols]

print("Modo sample (principal), época final:")
display(final_table_c("sample").round(3))
print("Modo argmax (secundario), época final:")
display(final_table_c("argmax").round(3))

### 6.5 Evaluación (ii): ¿obtiene la hidrofobicidad que se le pide?

Para cada objetivo `t` (percentiles 10, 30, 50, 70 y 90 del GRAVY real) se piden 2000 secuencias con esa condición (el mismo `z` para todos los objetivos). La tabla muestra la época 200 en modo sample (principal): GRAVY medio generado, error absoluto medio, tasa de acierto (±0,25), fracción de válidas y únicas, tamaño del subconjunto real de referencia (GRAVY en `[t - 0,25, t + 0,25]`) y, como propiedades secundarias, la W1 de longitud y de carga neta frente a ese subconjunto. Debajo, la **referencia sin condición** (la muestra de `gan_stabc`: no se puede dirigir, así que su GRAVY medio es el mismo para todo `t`) y el **techo real**.

In [ ]:
HCOLS = ["hydrophobicity_mean", "hydrophobicity_mae", "hit_rate", "valid_fraction", "unique_fraction", "n_reference",
         "length_wasserstein", "net_charge_wasserstein"]

def targets_h_at(mode, epoch=EPOCHS):
    sel = res_h.targets[(res_h.targets["epoch"] == epoch) & (res_h.targets["mode"] == mode)]
    return sel.set_index("target").loc[list(H_TARGETS)]

href_tab = unconditional_target_table(ref_seqs[REF_NAME]["sample"], H_TARGETS, H_TOL, prop="hydrophobicity").set_index("target")
small_h = [round(t, 3) for t in H_TARGETS if res_h.targets.loc[res_h.targets["target"] == t, "n_reference"].iloc[0] < 30]
print(f"Objetivos con menos de 30 secuencias reales de referencia (W1 = NaN): {small_h if small_h else 'ninguno'}.\n")

fin_hs = targets_h_at("sample")
tab_hs = fin_hs[HCOLS].copy()
tab_hs["GRAVY medio referencia"] = href_tab["hydrophobicity_mean"]
tab_hs["tasa de acierto referencia"] = href_tab["hit_rate"]
tab_hs["techo real"] = h_ceiling
print(f"Evaluación por objetivos, modo sample, época {EPOCHS} (n = {N_EVAL} por objetivo):")
display(tab_hs.round(3))
print(f"Referencia sin condición (gan_stabc, {len(ref_seqs[REF_NAME]['sample'])} secuencias): GRAVY medio {fmt(href_tab['hydrophobicity_mean'].iloc[0])}; "
      f"la condicional pide t = {', '.join(fmt(t) for t in H_TARGETS)}.")

fin_ha = targets_h_at("argmax")
tab_ha = fin_ha[HCOLS].copy()
tab_ha["tasa de acierto referencia"] = href_tab["hit_rate"]
tab_ha["techo real"] = h_ceiling
print(f"\nModo argmax (secundario), época {EPOCHS}:")
display(tab_ha.round(3))

**GRAVY generado frente a GRAVY pedido** (época final). Cada punto es el GRAVY medio de las secuencias pedidas con ese objetivo y la barra, su desviación típica. La línea discontinua gris es la **diagonal ideal** (generado = pedido), la línea marrón la regresión lineal de los cinco puntos y la horizontal azul el GRAVY medio de la referencia sin condición, que no depende de lo que se pida. La banda gris clara alrededor de la diagonal es la tolerancia de acierto (±0,25).

In [ ]:
def gen_gravy(mode, t):
    v = property_values(res_h.target_sequences[mode][t], "hydrophobicity")
    return v[~np.isnan(v)]

tt_h = np.array(H_TARGETS, dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
for ax, mode in zip(axes, ("sample", "argmax")):
    means = np.array([gen_gravy(mode, t).mean() for t in H_TARGETS])
    stds = np.array([gen_gravy(mode, t).std(ddof=1) for t in H_TARGETS])
    slope_m, icpt_m = control_slope(H_TARGETS, means)
    lim = [min(tt_h.min(), (means - stds).min()) - 0.3, max(tt_h.max(), (means + stds).max()) + 0.3]
    ax.fill_between(lim, [v - H_TOL for v in lim], [v + H_TOL for v in lim], color="gray", alpha=0.12)
    ax.plot(lim, lim, color="gray", ls="--", label="diagonal ideal (generado = pedido)")
    ax.axhline(href_tab["hydrophobicity_mean"].iloc[0], color=COLORS[REF_NAME], ls="-.",
               label=f"referencia sin condición (media {fmt(href_tab['hydrophobicity_mean'].iloc[0], 2)})")
    ax.plot(lim, [slope_m * v + icpt_m for v in lim], color="tab:brown", alpha=0.5, label=f"regresión: pendiente {slope_m:.2f}".replace(".", ","))
    ax.errorbar(tt_h, means, yerr=stds, fmt="o", color="tab:brown", capsize=3, label=f"{COND_H} (media +- desv. típica)")
    ax.set_title(f"GRAVY generado frente a pedido - {mode} (época {EPOCHS})")
    ax.set_xlabel("GRAVY pedido")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("GRAVY generado")
fig.tight_layout()
plt.show()

**Tasa de acierto (±0,25 de GRAVY) por objetivo.** Barras: techo real (fracción de datos reales dentro de la tolerancia, lo que daría una muestra con la distribución real sin dirigir), la referencia sin condición (`gan_stabc`) y la condicional por hidrofobicidad (sample y argmax).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
xs = np.arange(len(H_TARGETS))
w_bar = 0.2
series_hr = [("techo real", h_ceiling.loc[list(H_TARGETS)].to_numpy(), "gray", None),
             (REF_NAME, href_tab["hit_rate"].loc[list(H_TARGETS)].to_numpy(), COLORS[REF_NAME], None),
             (f"{COND_H}, sample", fin_hs["hit_rate"].to_numpy(), "tab:brown", None),
             (f"{COND_H}, argmax", fin_ha["hit_rate"].to_numpy(), "tab:brown", "//")]
for i, (label, vals, color, hatch) in enumerate(series_hr):
    bars = ax.bar(xs + (i - 1.5) * w_bar, vals, width=w_bar, color=color, hatch=hatch, alpha=0.9 if hatch is None else 0.5, label=label)
    for rect, v in zip(bars, vals):
        ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(xs)
ax.set_xticklabels([f"t = {t:.3f}\n(p{p})" for t, p in zip(H_TARGETS, TARGET_PERCENTILES)])
ax.set_ylabel(f"Tasa de acierto (|GRAVY - t| <= {H_TOL})")
ax.set_title(f"Tasa de acierto por GRAVY pedido (época {EPOCHS})")
ax.set_ylim(0, 1.12)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

### 6.6 Evolución por instantánea

**Evaluación (i).** Las mismas métricas que en 5.8 del notebook 03 y 6.6 del notebook 04 para la referencia, la WGAN-GP y la condicional por hidrofobicidad (continua: sample; discontinua: argmax); el último panel muestra el GRAVY medio en lugar de la longitud media. La línea gris punteada es el suelo `real_subset` (en el GRAVY medio, la media real).

In [ ]:
SHOWN_H = [REF_NAME, "WGAN-GP", COND_H]
panels_h = panels[:5] + [("hydrophobicity_mean", "GRAVY medio", float(props_real["hydrophobicity"].mean()))]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
for ax, (col, title, real_value) in zip(axes.ravel(), panels_h):
    for name in SHOWN_H:
        for mode, ls in (("sample", "-"), ("argmax", "--")):
            sel = METRICS_C[name][METRICS_C[name]["mode"] == mode]
            ax.plot(sel["epoch"], sel[col], ls, marker="o", ms=3.5, color=COLORS_C[name])
    ax.axhline(real_value if real_value is not None else floors["real_subset"][col], color="gray", ls=":")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.set_ylabel(title)
handles = [Line2D([], [], color=COLORS_C[n_], ls=ls_, marker="o", ms=3.5, label=f"{n_}, {m_}")
           for n_ in SHOWN_H for m_, ls_ in (("sample", "-"), ("argmax", "--"))]
handles.append(Line2D([], [], color="gray", ls=":", label="suelo real_subset (media real en el GRAVY medio)"))
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

real_mean_gravy = float(props_real["hydrophobicity"].mean())
snap_h = pd.concat({
    "W1 hidrofobicidad": pd.DataFrame({n: snap_series(METRICS_C[n], "hydrophobicity_wasserstein") for n in SHOWN_H}),
    f"GRAVY medio (real = {real_mean_gravy:.3f})": pd.DataFrame({n: snap_series(METRICS_C[n], "hydrophobicity_mean") for n in SHOWN_H}),
}, axis=1)
snap_h.index.name = "época"
print(f"W1 de hidrofobicidad y GRAVY medio por instantánea (modo sample); suelo real_subset de la W1 de hidrofobicidad: "
      f"{floors['real_subset']['hydrophobicity_wasserstein']:.3f}")
display(snap_h.round(3))

**Evaluación (ii).** Por instantánea y objetivo (modo sample): GRAVY medio generado (con la línea del objetivo), error absoluto medio y tasa de acierto. Debajo, la pendiente de la regresión del GRAVY medio generado sobre el pedido en cada instantánea (1 = control perfecto, 0 = el GRAVY no depende de la condición).

In [ ]:
cmap_t = plt.get_cmap("viridis")
tcs_h = res_h.targets[res_h.targets["mode"] == "sample"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
for i, t in enumerate(H_TARGETS):
    s_t = tcs_h[tcs_h["target"] == t]
    color = cmap_t(i / max(len(H_TARGETS) - 1, 1))
    lab = f"t = {t:.3f}"
    axes[0].plot(s_t["epoch"], s_t["hydrophobicity_mean"], marker="o", ms=3.5, color=color, label=lab)
    axes[0].axhline(t, color=color, ls=":", lw=1)
    axes[1].plot(s_t["epoch"], s_t["hydrophobicity_mae"], marker="o", ms=3.5, color=color, label=lab)
    axes[2].plot(s_t["epoch"], s_t["hit_rate"], marker="o", ms=3.5, color=color, label=lab)
for ax, title in zip(axes, ("GRAVY medio generado (punteado: objetivo)", "Error absoluto medio |GRAVY - t|",
                            f"Tasa de acierto (|GRAVY - t| <= {H_TOL})")):
    ax.set_title(title)
    ax.set_xlabel("Época")
axes[2].legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

pend_h = pd.DataFrame({
    mode: {e: control_slope(H_TARGETS, g.set_index("target").loc[list(H_TARGETS), "hydrophobicity_mean"])[0]
           for e, g in res_h.targets[res_h.targets["mode"] == mode].groupby("epoch")}
    for mode in ("sample", "argmax")})
pend_h.index.name = "época"
print("Pendiente del GRAVY medio generado frente al pedido, por instantánea:")
display(pend_h.T.round(3))

### 6.7 Distribuciones

**Evaluación (i), época final.** Histogramas normalizados de longitud, carga neta e hidrofobicidad: real (gris), referencia (azul) y condicional por hidrofobicidad (marrón), con los mismos intervalos y ejes en cada panel. Arriba, modo sample; abajo, modo argmax. Solo se usan las secuencias generadas válidas.

In [ ]:
valid_h = {name: {mode: [s for s in SEQS_C[name][mode] if is_valid(s)] for mode in MODES} for name in SEQS_C}
prof_h = {name: {mode: profile_sequences(v) if v else None for mode, v in d.items()} for name, d in valid_h.items()}
SHOWN_HD = [REF_NAME, COND_H]

edges_h = {}
for prop, _ in PROPS:
    allv = np.concatenate([prof_real.values(prop)] + [prof_h[n][m].values(prop) for n in SHOWN_HD for m in MODES if prof_h[n][m] is not None])
    lo, hi = np.floor(allv.min()), np.ceil(allv.max())
    edges_h[prop] = np.arange(lo - 0.5, hi + 1.5) if prop == "length" else np.linspace(lo, hi, 41)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="col", sharey="col")
for r, mode in enumerate(("sample", "argmax")):
    for c, (prop, xlabel) in enumerate(PROPS):
        ax = axes[r, c]
        real = prof_real.values(prop)
        ax.hist(real, bins=edges_h[prop], weights=np.full(real.size, 1 / real.size), color="gray", alpha=0.4, label=f"real (n={real.size:,})")
        for name in SHOWN_HD:
            p = prof_h[name][mode]
            if p is not None:
                g = p.values(prop)
                ax.hist(g, bins=edges_h[prop], weights=np.full(g.size, 1 / g.size), histtype="step", lw=1.6, color=COLORS_C[name],
                        label=f"{name} (n={g.size:,})")
        ax.set_title(f"{xlabel} - {mode}")
        ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel("Fracción de secuencias")
        ax.legend(fontsize=7, loc="upper right")
for c in range(3):
    axes[0, c].set_ylim(0, max(axes[r, c].get_ylim()[1] for r in range(2)) * 1.3)
fig.tight_layout()
plt.show()

**Evaluación (ii): distribución del GRAVY generado para cada GRAVY pedido** (modo sample, época final). La línea vertical marca el objetivo y la banda, la tolerancia de acierto (±0,25); en gris, el GRAVY real (todas las secuencias). Solo se usan las secuencias no vacías.

In [ ]:
fig, axes = plt.subplots(1, len(H_TARGETS), figsize=(3.4 * len(H_TARGETS), 3.6), sharey=True)
bins_g = np.arange(-4.6, 4.7, 0.1)
for i, (ax, t) in enumerate(zip(np.atleast_1d(axes), H_TARGETS)):
    g = gen_gravy("sample", t)
    ax.hist(real_gravy, bins=bins_g, weights=np.full(real_gravy.size, 1 / real_gravy.size), color="gray", alpha=0.35, label="real")
    ax.hist(g, bins=bins_g, weights=np.full(g.size, 1 / g.size), histtype="stepfilled", color=cmap_t(i / max(len(H_TARGETS) - 1, 1)), alpha=0.7,
            label=f"pedido t = {t:.3f}")
    ax.axvspan(t - H_TOL, t + H_TOL, color="k", alpha=0.08)
    ax.axvline(t, color="k", lw=1)
    ax.set_xlim(-4.0, 3.5)
    ax.set_title(f"t = {t:.3f}: acierto {fin_hs.loc[t, 'hit_rate']:.2f}".replace(".", ","))
    ax.set_xlabel("GRAVY generado")
    ax.legend(fontsize=7)
np.atleast_1d(axes)[0].set_ylabel("Fracción de secuencias")
fig.tight_layout()
plt.show()

### 6.8 Composición de aminoácidos

Fracción media de cada aminoácido en las secuencias reales y en las generadas válidas de la referencia y de la condicional por hidrofobicidad (evaluación (i)), ordenados por abundancia en las reales, con el mismo eje vertical en los dos modos.

In [ ]:
SHOWN_HC = [REF_NAME, COND_H]
comp_h = pd.DataFrame({"real": prof_real.composition}, index=list(STANDARD_ALPHABET))
for mode in MODES:
    for name in SHOWN_HC:
        comp_h[f"{name}|{mode}"] = prof_h[name][mode].composition if prof_h[name][mode] is not None else np.nan
comp_h = comp_h.sort_values("real", ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, sharey=True)
x_aa_h = np.arange(len(comp_h))
for ax, mode in zip(axes, ("sample", "argmax")):
    series = [("real", "real", "gray")] + [(f"{n}|{mode}", n, COLORS_C[n]) for n in SHOWN_HC]
    for i, (col, label, color) in enumerate(series):
        ax.bar(x_aa_h + (i - 1) * 0.27, comp_h[col] * 100, width=0.27, color=color, label=label)
    ax.set_ylabel("Composición media (%)")
    ax.set_title(f"Composición media de aminoácidos - {mode} (época final)")
    ax.legend()
axes[-1].set_xticks(x_aa_h)
axes[-1].set_xticklabels(comp_h.index)
axes[-1].set_xlabel("Aminoácido (ordenados por abundancia en las secuencias reales)")
fig.tight_layout()
plt.show()

### 6.9 Secuencias de ejemplo

Cinco secuencias pedidas con GRAVY igual a los percentiles 10, 50 y 90 (modo sample, época final), con su longitud, carga neta e hidrofobicidad. Las no válidas aparecen sin propiedades. Debajo, el recuento de repetición de la evaluación (i) (secuencias válidas, distintas entre ellas y veces que aparece la más repetida).

In [ ]:
for p in (10, 50, 90):
    t = H_TARGETS[TARGET_PERCENTILES.index(p)]
    print(f"GRAVY pedido t = {t:.3f} (percentil {p}), modo sample:".replace(".", ","))
    ex = example_table(res_h.target_sequences["sample"][t], k=5)
    ex.insert(0, "pedido", round(t, 3))
    display(ex)

rep_h = []
for name in (REF_NAME, COND_H):
    for mode in ("sample", "argmax"):
        valid = [s for s in SEQS_C[name][mode] if is_valid(s)]
        cnt = Counter(valid)
        top_seq, top_n = cnt.most_common(1)[0] if cnt else ("", 0)
        rep_h.append({"variante": name, "modo": mode, "generadas": len(SEQS_C[name][mode]), "válidas": len(valid),
                      "distintas": len(cnt), "más repetida (veces)": top_n, "secuencia más repetida": top_seq[:40]})
print("Recuentos de repetición, evaluación (i) (época final):")
display(pd.DataFrame(rep_h).set_index(["variante", "modo"]))

### 6.10 Comparación con la ejecución del script

El mismo experimento se ejecutó con `python src/amp_gan/run_conditional_gan.py --variant cgan_hydro --auto-backend` y sus resultados están en `context/data/processed/cgan_hydro_*`. Si existen, se comparan con los de esta ejecución (época final, modo sample): la evaluación (i) y la tasa de acierto y el GRAVY medio por objetivo de la evaluación (ii). Las diferencias, si las hay, proceden del backend o del dispositivo, no de la configuración (misma semilla y mismos parámetros).

In [ ]:
f_script_h = PROCESSED / "cgan_hydro_metrics.csv"
if f_script_h.exists():
    m_script_h = pd.read_csv(f_script_h)
    m_script_h = m_script_h[m_script_h["epoch"].notna()].astype({"epoch": int})
    info_script_h = json.loads((PROCESSED / "cgan_hydro_run_info.json").read_text(encoding="utf-8"))
    print(f"Script: backend {info_script_h['keras_backend']} en {info_script_h['device_used']}; esta ejecución: {env['keras_backend']} en {decision['device']}.")
    display(pd.DataFrame({"script": pick(m_script_h, "sample"), "esta ejecución": pick(res_h.metrics, "sample")}).loc[cols].round(3))
    t_script_h = pd.read_csv(PROCESSED / "cgan_hydro_targets.csv")
    t_script_h = t_script_h[(t_script_h["epoch"] == EPOCHS) & (t_script_h["mode"] == "sample")].sort_values("target")
    cmp_h = pd.DataFrame({"t": [round(t, 3) for t in H_TARGETS]})
    for col, label in (("hydrophobicity_mean", "GRAVY medio"), ("hit_rate", "tasa de acierto")):
        cmp_h[f"{label} (script)"] = t_script_h[col].to_numpy()
        cmp_h[f"{label} (esta ejecución)"] = fin_hs[col].to_numpy()
    print("Evaluación por objetivos (sample, época final):")
    display(cmp_h.set_index("t").round(3))
else:
    print("No existe cgan_hydro_metrics.csv (genéralo con python src/amp_gan/run_conditional_gan.py --variant cgan_hydro).")

### 6.11 Revisión del criterio A.2 (posterior al resultado)

El criterio inicial A.2 (error absoluto medio promediado sobre los cinco objetivos ≤ 0,30) **no se cumplió**. Al revisar los resultados se vio que ese umbral no mide si el modelo controla la hidrofobicidad, sino la dispersión natural del GRAVY de las secuencias generadas: si en cada objetivo el GRAVY generado tiene desviación típica σ y es aproximadamente normal, el error absoluto medio es ≈ 0,8·σ (`σ·√(2/π)`) aunque la media generada sea exacta, es decir, aunque el control de la media fuese perfecto. Con σ ≈ 0,7 el umbral de 0,30 era **inalcanzable por construcción**.

Se sustituye, **después de conocer el resultado** (no es un criterio fijado de antemano), por el **criterio A.2 revisado**: el error absoluto medio del modelo es menor que el de la referencia sin condición (`gan_stabc`, que no se puede dirigir) en cada uno de los cinco objetivos, sin umbral numérico. La tabla muestra, por objetivo (modo sample, época 200), el error del modelo y el de la referencia y, como magnitudes **solo descriptivas** (no son criterio), el sesgo `|media generada - t|` y la desviación típica del GRAVY generado. El cálculo está en `mae_vs_reference` y `target_dispersion_table` (`src/amp_gan/conditions.py`), reutilizables en el experimento con ambas condiciones.

In [ ]:
from amp_gan.conditions import mae_vs_reference, target_dispersion_table

A2_INICIAL_MAX = 0.30  # umbral del criterio inicial (fijado de antemano)
a2_info = mae_vs_reference(fin_hs, href_tab, "hydrophobicity", initial_max=A2_INICIAL_MAX)
disp_h = target_dispersion_table(res_h.target_sequences["sample"], H_TARGETS, "hydrophobicity")

tabla_a2 = pd.DataFrame({
    "error del modelo": a2_info["per_target"]["model_mae"],
    "error de la referencia": a2_info["per_target"]["reference_mae"],
    "¿menor?": a2_info["per_target"]["beats"],
    "sesgo |media - t|": disp_h["bias"],
    "desv. típica generada": disp_h["std"],
})
promedio = tabla_a2[["error del modelo", "error de la referencia", "sesgo |media - t|", "desv. típica generada"]].mean()
tabla_a2["¿menor?"] = tabla_a2["¿menor?"].map({True: "sí", False: "no"})
tabla_a2.index = [fmt(t) for t in tabla_a2.index]
tabla_a2.index.name = "objetivo t"
tabla_a2.loc["promedio"] = [promedio.iloc[0], promedio.iloc[1], "", promedio.iloc[2], promedio.iloc[3]]
display(tabla_a2.round(3))

std_media = float(disp_h["std"].mean())
print(f"Criterio inicial A.2: error medio {fmt(a2_info['mean_model_mae'])} frente a un máximo de {fmt(A2_INICIAL_MAX, 2)}.")
print(f"Desviación típica media del GRAVY generado: {fmt(std_media)}; con la media exacta y distribución normal el error medio sería "
      f"σ·√(2/π) = {fmt(std_media * np.sqrt(2 / np.pi))} (el umbral inicial era {fmt(A2_INICIAL_MAX, 2)}).")
print(f"Sesgo medio |media generada - t| (descriptivo): {fmt(disp_h['bias'].mean())}; "
      f"error medio de la referencia sin condición: {fmt(a2_info['mean_reference_mae'])}.")

### 6.12 Conclusiones (descriptivas)

Primero se evalúan los criterios A y B (modo sample, época 200, sin elegir la mejor época), condición por condición y con las cifras de esta ejecución; el A.2 se muestra con el criterio inicial (fijado de antemano) y con el revisado (modificado después de conocer el resultado, ver 6.11). Después se describen, sin interpretar causas, las demás métricas, la evolución, los tiempos y el comportamiento del entrenamiento. El texto se genera con el código de la celda siguiente; los números llevan coma decimal.

In [ ]:
H_SLOPE_RANGE, H_MAE_MAX, H_UNIQUE_MIN = (0.8, 1.2), 0.30, 0.95
SI_NO_H = lambda ok: "SE CUMPLE" if ok else "NO se cumple"

print("CRITERIO A - control de la hidrofobicidad (modo sample, época 200)")
slope_h, icpt_h = control_slope(H_TARGETS, fin_hs["hydrophobicity_mean"])
a1 = H_SLOPE_RANGE[0] <= slope_h <= H_SLOPE_RANGE[1]
print(f"  1. Pendiente del GRAVY medio generado frente al pedido: {fmt(slope_h)} (ordenada {fmt(icpt_h)}) -> {SI_NO_H(a1)} "
      f"(se pedía entre {fmt(H_SLOPE_RANGE[0], 1)} y {fmt(H_SLOPE_RANGE[1], 1)}).")
a2_info = mae_vs_reference(fin_hs, href_tab, "hydrophobicity", initial_max=H_MAE_MAX)
a2_ini, a2 = a2_info["initial_ok"], a2_info["ok"]
print(f"  2. (criterio INICIAL, fijado de antemano) Error absoluto medio |GRAVY - t| promediado sobre los objetivos: {fmt(a2_info['mean_model_mae'])} "
      f"-> {SI_NO_H(a2_ini)} (se pedía <= {fmt(H_MAE_MAX, 2)}; inalcanzable por la dispersión del GRAVY generado, ver 6.11).")
print("  2'. (criterio REVISADO, modificado DESPUÉS de conocer el resultado) El error absoluto medio del modelo es menor que el de la referencia en cada objetivo:")
for t, r in a2_info["per_target"].iterrows():
    print(f"     t = {fmt(t)}: error del modelo {fmt(r['model_mae'])} frente a referencia {fmt(r['reference_mae'])} -> {'menor' if r['beats'] else 'NO menor'}")
print(f"     Menor en {a2_info['n_beaten']} de {a2_info['n_targets']} objetivos -> {SI_NO_H(a2)}.")
a3 = []
for t in H_TARGETS:
    ok = bool(fin_hs.loc[t, "hit_rate"] > href_tab.loc[t, "hit_rate"])
    a3.append(ok)
    print(f"     t = {fmt(t)}: tasa de acierto condicional {fmt(fin_hs.loc[t, 'hit_rate'])} frente a referencia {fmt(href_tab.loc[t, 'hit_rate'])} "
          f"(techo real {fmt(h_ceiling[t])}) -> {'supera' if ok else 'NO supera'}")
a3_ok = all(a3)
print(f"  3. Tasa de acierto superior a la de la referencia en los {len(H_TARGETS)} objetivos: la supera en {sum(a3)} de {len(H_TARGETS)} -> {SI_NO_H(a3_ok)}.")
A_ok_h = a1 and a2 and a3_ok
A_ini_h = a1 and a2_ini and a3_ok
print(f"  Resultado A con el criterio INICIAL: {sum([a1, a2_ini, a3_ok])} de 3 condiciones -> la condicional {'SÍ' if A_ini_h else 'NO'} controla la hidrofobicidad.")
print(f"  Resultado A con el criterio REVISADO (A.2 modificado después del resultado): {sum([a1, a2, a3_ok])} de 3 condiciones -> la condicional {'SÍ' if A_ok_h else 'NO'} controla la hidrofobicidad.")
print(f"  Descriptivo (no es criterio): sesgo medio |media generada - t| {fmt(disp_h['bias'].mean())} (por objetivo: "
      + "; ".join(f"t={fmt(t)}: {fmt(v)}" for t, v in disp_h['bias'].items()) + "); desviación típica generada por objetivo: "
      + "; ".join(f"t={fmt(t)}: {fmt(v)}" for t, v in disp_h['std'].items()) + ".\n")

h_s, hr_s = pick(res_h.metrics, "sample"), pick(ALL_METRICS[REF_NAME], "sample")
hl_s, hw_s = pick(ALL_METRICS[COND_NAME], "sample"), pick(ALL_METRICS["WGAN-GP"], "sample")
print("CRITERIO B - fidelidad con condiciones reales (evaluación (i), modo sample, época 200)")
b1 = bool(h_s["hydrophobicity_wasserstein"] <= hr_s["hydrophobicity_wasserstein"])
print(f"  1. W1 de hidrofobicidad: referencia {fmt(hr_s['hydrophobicity_wasserstein'], 4)}, condicional {fmt(h_s['hydrophobicity_wasserstein'], 4)} "
      f"-> {SI_NO_H(b1)} (se pedía <= la de la referencia).")
b2 = bool(h_s["unique_fraction"] >= H_UNIQUE_MIN)
print(f"  2. unique_fraction: referencia {fmt(hr_s['unique_fraction'])}, condicional {fmt(h_s['unique_fraction'])} -> {SI_NO_H(b2)} (se pedía >= {fmt(H_UNIQUE_MIN, 2)}).")
B_ok_h = b1 and b2
print(f"  Resultado B: {sum([b1, b2])} de 2 condiciones -> la condicional {'NO empeora' if B_ok_h else 'SÍ empeora (según el criterio)'} a la referencia.")
print("  Informativas, sin condición de aprobado (condicional por hidrofobicidad; entre paréntesis, la referencia; entre corchetes, la condicional por longitud):")
for col, label in (("length_wasserstein", "W1 de longitud"), ("net_charge_wasserstein", "W1 de carga neta"),
                   ("composition_tv", "variación total de composición")):
    print(f"     {label}: {fmt(h_s[col])} ({fmt(hr_s[col])}) [{fmt(hl_s[col])}]")
print("\nEs una sola ejecución con la semilla 42: no se estima la variabilidad entre semillas ni se ajusta ningún hiperparámetro.")

print()
print(f"Rango habitual p10-p90: de {fmt(H_LO)} a {fmt(H_HI)}; objetivos {', '.join(fmt(t) for t in H_TARGETS)}; tolerancia +-{fmt(H_TOL, 2)}.")
print("Secuencias reales con GRAVY dentro de la tolerancia, por objetivo: " + "; ".join(f"t={fmt(t)}: {h_near[t]}" for t in H_TARGETS)
      + f" (mínimo {min(h_near.values())}; la W1 secundaria exige al menos 30, ver la tabla 6.5).")
ext = [(p, t) for p, t in zip(TARGET_PERCENTILES, H_TARGETS) if p in (10, 90)]
for p, t in ext:
    gm = fin_hs.loc[t, "hydrophobicity_mean"]
    print(f"  Extremo p{p} (t = {fmt(t)}): GRAVY medio generado {fmt(gm)} ({'por encima' if gm > t else 'por debajo'} en {fmt(abs(gm - t))}), "
          f"tasa de acierto {fmt(fin_hs.loc[t, 'hit_rate'])} (referencia {fmt(href_tab.loc[t, 'hit_rate'])}, techo real {fmt(h_ceiling[t])}), "
          f"{h_near[t]} secuencias reales cerca.")
print(f"[sample] época {EPOCHS}, evaluación (i): condicional por hidrofobicidad (referencia; suelos real_subset / random_uniform)")
fl, ru = floors["real_subset"], floors["random_uniform"]
for col, label in (("valid_fraction", "válidas"), ("unique_fraction", "únicas"), ("novel_fraction", "nuevas"),
                   ("length_mean", "longitud media"), ("length_wasserstein", "W1 longitud"),
                   ("net_charge_mean", "carga media"), ("net_charge_wasserstein", "W1 carga"),
                   ("hydrophobicity_mean", "GRAVY medio"), ("hydrophobicity_wasserstein", "W1 hidrofobicidad"),
                   ("composition_tv", "variación total de composición")):
    suelos = f"{fmt(fl[col])} / {fmt(ru[col])}" if col in fl else "-"
    print(f"  - {label}: {fmt(h_s[col])} ({fmt(hr_s[col])}); suelos {suelos}")
h_a, hr_a = pick(res_h.metrics, "argmax"), pick(ALL_METRICS[REF_NAME], "argmax")
print(f"[argmax] época {EPOCHS}: unique_fraction {fmt(h_a['unique_fraction'])} ({fmt(hr_a['unique_fraction'])}); "
      f"W1 hidrofobicidad {fmt(h_a['hydrophobicity_wasserstein'], 4)} ({fmt(hr_a['hydrophobicity_wasserstein'], 4)}); "
      f"valid_fraction {fmt(h_a['valid_fraction'])} ({fmt(hr_a['valid_fraction'])}).")

print(f"\nEvaluación por objetivos (sample, época {EPOCHS}): GRAVY medio generado " + "; ".join(f"t={fmt(t)}: {fmt(fin_hs.loc[t, 'hydrophobicity_mean'])}" for t in H_TARGETS) + ".")
print("  unique_fraction por objetivo: " + "; ".join(f"t={fmt(t)}: {fmt(fin_hs.loc[t, 'unique_fraction'])}" for t in H_TARGETS)
      + ". valid_fraction: " + "; ".join(f"t={fmt(t)}: {fmt(fin_hs.loc[t, 'valid_fraction'])}" for t in H_TARGETS) + ".")
print(f"  Evaluación por objetivos (argmax): pendiente {fmt(control_slope(H_TARGETS, fin_ha['hydrophobicity_mean'])[0])}, "
      f"error absoluto medio {fmt(fin_ha['hydrophobicity_mae'].mean())}, tasas de acierto "
      + "; ".join(f"t={fmt(t)}: {fmt(fin_ha.loc[t, 'hit_rate'])}" for t in H_TARGETS) + ".")
print("  Propiedades secundarias (sample), W1 de longitud frente al subconjunto real: " + "; ".join(f"t={fmt(t)}: {fmt(fin_hs.loc[t, 'length_wasserstein'], 2)}" for t in H_TARGETS)
      + ". W1 de carga neta: " + "; ".join(f"t={fmt(t)}: {fmt(fin_hs.loc[t, 'net_charge_wasserstein'], 2)}" for t in H_TARGETS) + ".")

print()
print("Pendiente por instantánea (sample): " + "; ".join(f"{e}: {fmt(v, 2)}" for e, v in pend_h["sample"].items()) + ".")
w1h = snap_series(res_h.metrics, "hydrophobicity_wasserstein")
w1hr = snap_series(ALL_METRICS[REF_NAME], "hydrophobicity_wasserstein")
print("W1 de hidrofobicidad por instantánea (sample), condicional: " + "; ".join(f"{e}: {fmt(v)}" for e, v in w1h.items()) + ".")
print(f"Entre las épocas 20 y {EPOCHS}, la W1 de hidrofobicidad de la condicional va de {fmt(w1h[w1h.index >= 20].max())} (máximo) a {fmt(w1h[w1h.index >= 20].min())} (mínimo); "
      f"la de la referencia, de {fmt(w1hr[w1hr.index >= 20].max())} a {fmt(w1hr[w1hr.index >= 20].min())}.")

f5h, l5h = hist_h.head(5).mean(), hist_h.tail(5).mean()
print(f"\nPérdidas medias (primeras 5 -> últimas 5 épocas): discriminador real {fmt(f5h['d_loss_real'])} -> {fmt(l5h['d_loss_real'])}; "
      f"discriminador falso {fmt(f5h['d_loss_fake'])} -> {fmt(l5h['d_loss_fake'])}; generador {fmt(f5h['g_loss'])} -> {fmt(l5h['g_loss'])}.")
hr5 = ref_hist[REF_NAME].tail(5).mean()
print(f"Referencia (últimas 5 épocas): discriminador real {fmt(hr5['d_loss_real'])}; falso {fmt(hr5['d_loss_fake'])}; generador {fmt(hr5['g_loss'])}.")
print("Valores no finitos en el historial:", int((~np.isfinite(hist_h.to_numpy(dtype=float))).sum()),
      "; NaN en las métricas (i):", int(res_h.metrics.drop(columns=["epoch", "mode"]).isna().sum().sum()),
      "; NaN en la evaluación por objetivos (aparte de las W1 con subconjunto pequeño):",
      int(res_h.targets.drop(columns=["epoch", "mode", "length_wasserstein", "net_charge_wasserstein"]).isna().sum().sum()), ".")
rch = pd.DataFrame(rep_h)
for name in (COND_H, REF_NAME):
    sub = rch[rch["variante"] == name]
    print(f"{name}: " + "; ".join(f"[{d['modo']}] {d['distintas']} secuencias distintas de {d['válidas']} válidas "
                                    f"(la más repetida, {d['más repetida (veces)']} veces)" for _, d in sub.iterrows()) + ".")
print(f"Tiempo: condicional por hidrofobicidad {fmt(t_h['mean_epoch_s'], 2)} s/época de media ({t_h['total_s']:.0f} s en total, backend {env['keras_backend']} "
      f"en {decision['device']}); referencia {fmt(ref_info[REF_NAME]['timing_s']['mean_epoch_s'], 2)} s/época "
      f"({ref_info[REF_NAME]['keras_backend']} en {ref_info[REF_NAME]['device_used']}).")

### 6.13 Nota sobre tiempos

La condicional por hidrofobicidad tiene el mismo coste por época que la condicional por longitud, parecido al de la vanilla estabilizada con ruido constante (con torch en GPU, 2,52 s/época en la ejecución del script frente a 2,36 s de la referencia). A las evaluaciones por instantánea se suman las 5 tandas de 2000 secuencias por objetivo y modo, que siguen siendo una fracción pequeña del total (~8 s con torch en GPU en la ejecución del script y ~11 s con TensorFlow en CPU). Los tiempos medidos están en la nota de la introducción; la celda de conclusiones imprime los de esta ejecución.